<a href="https://colab.research.google.com/github/ryanmart25/bird-song-recognizer/blob/utils_setup/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bird Species Identifier
A model for identifying the species of bird from audio.

## Imports

In [37]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path
import os
import sys
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D, MaxPooling2D, concatenate, Conv1D, MaxPooling1D
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_curve, auc, mean_squared_error
import matplotlib.pyplot as plt
from collections.abc import Sequence
from sklearn import preprocessing
%matplotlib inline
import csv
import glob
from IPython.display import Image
import seaborn as sns


## Global Control Flow Flags and Program Configuration

In [35]:
ITERATION = 0
PAUL = True # paul, you are running in an environment with a different keras backend than us, and are using pytorch instead of tensorflow.
# use this flag to gate code that should be run when only you want it to run. If this feels like a clunky and bad idea, feel free to disregard this.
# In general, my idea for these flags was they could be used to section off highly experimental / broken code, or code that only works in a specific
# environment that others might not have.
RYAN = True
BEN = True
WINDOW_SIZE = 7
OPTIMIZER_LEARNING_RATE = 0.001


## Define Helper Methods

In [38]:
def plot_losses(history, base_path, iteration:int):
    # Plot training & validation loss over epochs
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.ylim(bottom=0.0, top=10.0)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs. Validation Loss")
    plt.legend()
    plt.savefig(
        os.path.join(base_path, f"training-validiation-loss--epoch---Model {iteration}")
    )
    plt.close()


def print_schema(dataframe: pd.DataFrame):
    print('~~~~~~dataframe schema~~~~~~')
    print(f"Dataframe shape: {dataframe.shape} | Dataframe length: {len(dataframe)}")
    print('Column labels: ')
    print(dataframe.columns)
    print('Dataframe head: ')
    print(f"{dataframe.head()}")
def print_column(dataframe: pd.DataFrame, columns: str | list[str]):
    if isinstance(columns, list):
        for i, label in enumerate(columns):
            print(f"column {i}")
            print(dataframe[label])
    else:
        print(dataframe[columns])
# Encode text values to dummy variables(i.e. [1,0,0],[0,1,0],[0,0,1] for red,green,blue)
def encode_text_dummy(df, name):
    dummies = pd.get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = "{}-{}".format(name, x)
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)


# Encode text values to indexes(i.e. [1],[2],[3] for red,green,blue).
def encode_text_index(df, name):
    le = preprocessing.LabelEncoder()
    df[name] = le.fit_transform(df[name])
    return le.classes_


# Encode a numeric column as zscores
def encode_numeric_zscore(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Convert all missing values in the specified column to the median
def missing_median(df, name):
    med = df[name].median()
    df[name] = df[name].fillna(med)


# Convert all missing values in the specified column to the default
def missing_default(df, name, default_value):
    df[name] = df[name].fillna(default_value)


# Convert a Pandas dataframe to the x,y inputs that TensorFlow needs
def to_xy(df, target):
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    # find out the type of the target column.
    target_type = df[target].dtypes
    target_type = target_type[0] if isinstance(target_type, Sequence) else target_type
    # Encode to int for classification, float otherwise. TensorFlow likes 32 bits.
    #if target_type in (np.int64, np.int32):
        ## Classification
        #dummies = pd.get_dummies(df[target])
        #return df[result].values.astype(np.float32), dummies.values.astype(np.float32)
    #else#:
        ## Regression
    return df[result].values.astype(np.float32), df[target].values.astype(np.float32)

# Nicely formatted time string
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Regression chart.
def chart_regression(path, pred,y,sort=True):
    t = pd.DataFrame({'pred' : pred, 'y' : y.flatten()})
    if sort:
        t.sort_values(by=['y'],inplace=True)
    b = plt.plot(t['pred'].tolist(),label='prediction')
    a = plt.plot(t['y'].tolist(),label='expected')

    plt.ylabel('output')
    plt.legend()
    plt.savefig(path)
    plt.close()

# Remove all rows where the specified column is +/- sd standard deviations
def remove_outliers(df, name, sd):
    drop_rows = df.index[(np.abs(df[name] - df[name].mean()) >= (sd * df[name].std()))]
    df.drop(drop_rows, axis=0, inplace=True)


# Encode a column to a range between normalized_low and normalized_high.
def encode_numeric_range(df, name, normalized_low=-1, normalized_high=1,
                         data_low=None, data_high=None):
    if data_low is None:
        data_low = min(df[name])
        data_high = max(df[name])

    df[name] = ((df[name] - data_low) / (data_high - data_low)) \
               * (normalized_high - normalized_low) + normalized_low



## Import and Read Datasets

In [41]:
# import an API key for the dataset
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ryanmartinez2025","key":"997d4c5c308e99f7d25bee0d94d11138"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json



In [43]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("ryanmartinez2025/birdcall-spectrograms")

print("Path to dataset files:", path)

100%|██████████| 53.2M/53.2M [00:03<00:00, 15.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ryanmartinez2025/bird-calls-spectrograms/versions/1


## Configure Environment

In [ ]:
import os
import sys
output_path = os.path.join(os.getcwd(), "output")
iteration_path= os.path.join(output_path, f"iteration-{ITERATION}")
base_dataset_path = path
# idk how you are going to seperate train/test/val. Are you seperating them into
# seperate .csv files or are we simply going to do 1 big csv file that we use
# train_test_split on?
train_ds_path = os.path.join(base_dataset_path, "train")
test_ds_path = os.path.join(base_dataset_path, "test")
val_ds_path = os.path.join(base_dataset_path, "val")
try:
  os.makedirs(iteration_path)
except FileExistsError as e:
  print(f"{iteration_path} already exists. exiting to save previous work.")
  sys.exit(0)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

checkpointer = ModelCheckpoint(filepath=os.path.join(full_path, "best_weights.keras"), verbose=0, save_best_only=True) #save best model
optimizer = Adam(learning_rate = 0.0001)
input_shape = (400,1000,1) #input shape H,W,Channels
num_classes = 2 #num_classes = classes to predict

inputs = layers.Input(shape=input_shape)
#CNN
#block 1
x = layers.Conv2D(32, kernel_size=3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#block 2
x = layers.Conv2D(64, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#block 3
x = layers.Conv2D(128, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#block 4
x = layers.Conv2D(256, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

#reshape for the LSTM
# -> (time, features)
shape = x.shape
x = layers.Reshape((shape[2], shape[1] * shape[3]))(x)

#LSTM for time modeling
x = layers.LSTM(128, return_sequences=True)(x) #output (62,128)
x = layers.Dropout(0.3)(x)
x = layers.LSTM(64, return_sequences=False)(x) #output (128)
x = layers.Dropout(0.3)(x)

#dense layers for classification
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.4)(x)

#output layer (variable number of classes to predict)
outputs = layers.Dense(num_classes, activation='softmax')(x)

#create the model
model = Model(inputs=inputs, outputs=outputs)

#model compilation
model.compile(optimizer=optimizer, loss='categorical_crossentropy')

monitor = EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=5, verbose=2, mode='min', restore_best_weights=True)

history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=100, batch_size=16, callbacks=[checkpointer, monitor])


In [ ]:
metrics_path = "metrics.txt"
def redirect(out): # redirect model summary to metrics
    with open(os.path.join(iteration_path, metrics_path), 'a') as file:
        print(out, file=file)
# make prediction and evaluate
model.load_weights(os.path.join(iteration_path, "best-weights.keras"))
prediction = model.predict(x_test)
score = np.sqrt(mean_squared_error(y_test, prediction))
if DEBUG:
    print("Score (RMSE): {}".format(score))
# Write Metrics to file
with open(os.path.join(iteration_path, metrics_path), "x") as file:
    file.write(f"Score (RMSE): {score}\n")
model.summary(print_fn=redirect)
chart_regression(os.path.join(iteration_path, "Lift-Chart"), prediction.flatten(), y_test, sort=True)